# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides an example workflow for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:
- [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json)

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print dataset metadata
meta = dataset.metadata
print(f"Dataset Name: {getattr(meta, 'name', 'N/A')}")
print(f"Description: {getattr(meta, 'description', 'N/A')}")
print(f"Version: {getattr(meta, 'version', 'N/A')}")
print(f"Published: {getattr(meta, 'datePublished', 'N/A')}")
print(f"License: {getattr(meta, 'license', 'N/A')}")

## 2. Data Overview
List all available record sets, their fields, and important column `@id`s for reference.

In [ ]:
# Discover record set information via schema
record_sets = list(dataset.metadata.record_sets)
print(f"Record Sets in dataset (with @id):")
for rset in record_sets:
    print(f"- Name: {getattr(rset, 'name', rset.id)}; @id: {rset.id}")
    print("  Fields/Columns:")
    for field in getattr(rset, 'fields', []):
        print(f"    - {getattr(field, 'name', field.id)} (@id: {field.id}, dtype: {getattr(field, 'data_type', 'N/A')})")
    print()
# We will use these @ids for extraction and analysis.

## 3. Data Extraction
Load data from record sets into pandas DataFrames for analysis. Use the `@id` fields from the overview above.

In [ ]:
# Extract records into DataFrames
# For the FAIR^2 dataset, main tabular data is typically in a primary record set. List all for completeness.
import collections

# Build a dict to hold DataFrames by record set @id
dataframes = collections.OrderedDict()

for rset in record_sets:
    print(f"Loading record set: {rset.id}")
    records = list(dataset.records(record_set=rset.id))  # Record set @id
    df = pd.DataFrame(records)
    print(f"Loaded {len(df)} records for {rset.id} with columns: {df.columns.tolist()}")
    dataframes[rset.id] = df
print("--- DataFrames available with record set @ids ---")
for k in dataframes:
    print(f"  {k} : shape={dataframes[k].shape}")
# Choose main data for further analysis (replace below @id if needed)
main_recordset_id = next(iter(dataframes.keys()))  # Use first if unsure
print(f"\nColumns in main record set ({main_recordset_id}):\n", dataframes[main_recordset_id].columns.tolist())
dataframes[main_recordset_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic processing (filtering, normalization, grouping) to a numeric field, using field `@id` for selection.

In [ ]:
# Select a numeric field for analysis: Use column @id or name as listed previously
# Example: For a protein dataset, typical numeric fields could be 'Abundance', 'MW', etc.
df = dataframes[main_recordset_id]
numeric_field = None
for col in df.columns:
    if 'abundance' in col.lower() or 'mw' in col.lower() or 'count' in col.lower() or 'coverage' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.select_dtypes('number').columns[0]  # Fallback to any numeric

print(f"Selected numeric field for EDA: {numeric_field}")

# Quick filter: Rows where value > threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f}: {len(filtered_df)} rows")
print(filtered_df[[numeric_field]].head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() or 1.0)
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field
cat_field = None
for col in df.columns:
    if col != numeric_field and df[col].nunique() < 15:
        cat_field = col
        break

if cat_field:
    print(f"Grouping by: {cat_field}")
    grouped_df = filtered_df.groupby(cat_field)[numeric_field].mean().reset_index()
    print(grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between selected fields.

If the notebook is running in a JupyterLab or notebook interface, you should see inline figures.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

# Histogram of numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.show()

# Boxplot of numeric field by group (if available)
if cat_field:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=cat_field, y=numeric_field, data=df)
    plt.title(f"{numeric_field} by {cat_field}")
    plt.xticks(rotation=30)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to use the `mlcroissant` library to load, explore, and process the FAIR^2 protein mass spectrometry dataset by referencing all entities by their Croissant schema `@id`. You explored available record sets, loaded tabular data, applied typical exploratory analysis and visualized protein-level metrics. For deeper insight or task-specific processing, continue by referencing field and record set `@id`s identified interactively above.